# MNIST learnable reduction — does a learned taper beat counting votes?

A copy of `mnist_grid_kaggle.ipynb` with one thing changed: the grid. Everything else — the
environment, the pinned upstream commit, binarization, `train_one`, `save_config` — is identical,
because **learnable reduction needs no new training code.**

## What learnable reduction is

A DWN normally ends with `GroupSum`: split the final layer's bits into one contiguous group per
class and count the ones. In hardware that count is an adder tree, the only arithmetic in an
otherwise arithmetic-free design, and on this project's JSC study it was measured at **35% of the
headline design** and sitting on the critical path.

The paper proposes replacing it with a *pyramid* of LUT layers that **learn** how to combine the
final bits. `torch_dwn` ships no such module — but it does not need to, because
`LUTLayer(input_size, output_size, ...)` takes its two widths independently. A taper is just more
layers:

```
1x2000    :  2000 bits ────────────────────► GroupSum(10), groups of 200
2x[2000,100]: 2000 → 100 bits ─────────────► GroupSum(10), groups of  10
```

**GroupSum still exists at the end, and that is not a compromise.** It returns continuous scores,
which is what cross-entropy trains against; a LUT layer returns bits. A pyramid that ran all the
way down to one output per class would give a single bit per class and no gradient to work with.
So the taper does not *replace* the reduction — it shrinks what the reduction has to count.

## The comparison

Every config is an existing grid model with taper layers appended. The network is unchanged, so
each row pairs against a baseline in `mnist_grid_kaggle.ipynb` and only the reduction differs.
**Train those baselines first**, or there is nothing to compare against.

Two questions the grid is built to answer:

1. **Does a taper hold accuracy?** If `2x[2000, 100]` matches `1x2000`, the popcount can shrink
   twentyfold for the cost of 100 extra nodes.
2. **Does score resolution bind?** Tapering to 100 leaves each class 10 bits — **11 distinct
   scores**. On JSC, where groups were also 10 bits, 29 of 1000 vectors tied for top and
   tie-breaking had to become part of the spec. The `resolution` group varies the floor (10, 20,
   50 bits per class) to find out whether that is what limits accuracy.

⚠️ **`tau` is held at upstream's `1/0.3` for every config**, exactly as the base grid does. With
one MNIST anchor there is no schedule to fit, and a config that changed both the architecture and
`tau` would confound the result — the JSC study lost a training run to a `tau` error that looked
like an architectural finding. Every final layer here is at least 100, so no group falls below the
size that anchor was measured at.

## Before you run anything

Same as the base notebook: **Accelerator → GPU**, **Internet → On**. To continue an unfinished
run, add the previous run's Output as an input dataset.

## What you get out

`<slug>_checkpoint.pt` and `<slug>_testvectors.npz` per config, in the format
`exporter/extract.py` reads. Download into `training/artifacts/mnist/`.


In [ ]:
# ---- environment check: fail loudly and early ----
import subprocess, sys, torch

print('torch     :', torch.__version__)
print('cuda avail:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu       :', torch.cuda.get_device_name(0))
    print('cuda (torch built against):', torch.version.cuda)
else:
    raise SystemExit('No GPU. Set Accelerator -> GPU in the settings panel. '
                     'DWN training cannot run on CPU.')

print()
print(subprocess.run(['nvcc', '--version'], capture_output=True, text=True).stdout)
print('If the nvcc CUDA version and the torch CUDA version differ a lot, the extension')
print('build in the next cell is where it will show up.')

In [ ]:
# ---- clone upstream DWN at the pinned commit and build the CUDA extension ----
# Pin matches third_party/DWN in the repo. Do not float this to main: the exporter is
# built against whatever checkpoint format this commit produces (CLAUDE.md).
PINNED_COMMIT = '9f887a0b4bd84dabf6d8c9ae35368ab2a7e0e3c0'

!rm -rf /kaggle/working/DWN
!git clone --quiet https://github.com/alanbacellar/DWN.git /kaggle/working/DWN
!cd /kaggle/working/DWN && git checkout --quiet {PINNED_COMMIT} && git log -1 --format='pinned at %h %ad %s'

# Confirm the CUDA sources actually exist at this pin before spending 5 minutes on a build.
# The pinned commit is literally "Delete custom_operators/cuda directory (Duplicate)" -- it
# removed a duplicate copy, not the real one, but that is worth verifying rather than assuming.
!ls -la /kaggle/working/DWN/src/torch_dwn/custom_operators/cuda/

# This compiles efd_cuda_kernel.cu with nvcc. Expect 2-5 minutes. It is the slowest and
# most fragile step in the notebook.
#
# --no-build-isolation is REQUIRED, not an optimization. Upstream's pyproject.toml declares
#     [build-system] requires = ["setuptools>=42", "wheel", "torch"]
# so a plain `pip install .` builds in a fresh isolated env and downloads ANOTHER torch from
# PyPI. setup.py's `import torch` then resolves to that one instead of the session's, so the
# extension gets built against a torch/CUDA pair that does not match the runtime -- it either
# fails to compile outright or builds and then fails to import on an ABI mismatch.
#
# Full output on purpose. Do NOT pipe this through `tail`: real compiler errors appear near
# the TOP of the log, while the last 20 lines are always the same generic pip epilogue
# ("did not run successfully / See above for output"), which identifies nothing.
!cd /kaggle/working/DWN && pip install --no-build-isolation .


In [ ]:
# ---- VERIFY the extension actually built ----
# This cell exists because the failure is otherwise silent. lut_layer.py does
#     if torch.cuda.is_available(): import efd_cuda
# so a failed build produces no error at install time -- you would instead get a bare
# NameError at the first forward pass, long after the real cause.
import torch, torch_dwn as dwn

try:
    import efd_cuda
    print('efd_cuda imported OK')
except ImportError as e:
    raise SystemExit(
        'efd_cuda failed to import -- the CUDA extension did not build.\n'
        'Scroll to the TOP of the install cell output and read the first compiler error;\n'
        'the tail of a pip failure is generic boilerplate and never names the cause.\n'
        f'Original error: {e}'
    )

# tiny end-to-end forward+backward, so we find out here rather than 200 lines later
_probe = torch.nn.Sequential(dwn.LUTLayer(12, 6, n=6), dwn.GroupSum(k=2, tau=1.0)).cuda()
_x = (torch.rand(4, 12, device='cuda') > 0.5).float()
_out = _probe(_x)
_out.sum().backward()
print('forward + backward OK, output shape', tuple(_out.shape))
del _probe, _x, _out


In [ ]:
# ---- the grid: learnable-reduction configurations ----
# Every config here is an existing grid model with TAPER LAYERS APPENDED. The network is
# unchanged; only what GroupSum counts is different. That is what makes each row a controlled
# comparison against a baseline in mnist_grid_kaggle.ipynb rather than a new model.
#
# `groupsum_group` and `note` are extra fields for reading the results. Nothing downstream reads
# them -- save_config() copies a fixed key list into the checkpoint, so they do not leak in.
TRAINING_SET = [
 {
  "slug": "mnist_n6_z3_distributive_w1000x100",
  "label": "2x[1000, 100]",
  "group": "taper-1",
  "n": 6,
  "thermometer_bits": 3,
  "layers": [
   1000,
   100
  ],
  "mapping": [
   "learnable",
   "learnable"
  ],
  "thermometer": "distributive",
  "num_classes": 10,
  "tau": 3.3333333333333335,
  "batch_size": 100,
  "epochs": 30,
  "lr": 0.01,
  "lr_step": 14,
  "lr_gamma": 0.1,
  "seed": 20260811,
  "groupsum_group": 10,
  "note": "one taper step from 1x1000. GroupSum sees 100 bits instead of 1000"
 },
 {
  "slug": "mnist_n6_z3_distributive_w2000x100",
  "label": "2x[2000, 100]",
  "group": "taper-1",
  "n": 6,
  "thermometer_bits": 3,
  "layers": [
   2000,
   100
  ],
  "mapping": [
   "learnable",
   "learnable"
  ],
  "thermometer": "distributive",
  "num_classes": 10,
  "tau": 3.3333333333333335,
  "batch_size": 100,
  "epochs": 30,
  "lr": 0.01,
  "lr_step": 14,
  "lr_gamma": 0.1,
  "seed": 20260811,
  "groupsum_group": 10,
  "note": "one taper step from 1x2000 -- the 20x popcount reduction, cheapest form"
 },
 {
  "slug": "mnist_n6_z3_distributive_w2000x300x100",
  "label": "3x[2000, 300, 100]",
  "group": "taper-2",
  "n": 6,
  "thermometer_bits": 3,
  "layers": [
   2000,
   300,
   100
  ],
  "mapping": [
   "learnable",
   "learnable",
   "learnable"
  ],
  "thermometer": "distributive",
  "num_classes": 10,
  "tau": 3.3333333333333335,
  "batch_size": 100,
  "epochs": 30,
  "lr": 0.01,
  "lr_step": 14,
  "lr_gamma": 0.1,
  "seed": 20260811,
  "groupsum_group": 10,
  "note": "two steps to the same 100. Does a gentler taper hold accuracy better than one jump?"
 },
 {
  "slug": "mnist_n6_z3_distributive_w2000x500x200x100",
  "label": "4x[2000, 500, 200, 100]",
  "group": "taper-3",
  "n": 6,
  "thermometer_bits": 3,
  "layers": [
   2000,
   500,
   200,
   100
  ],
  "mapping": [
   "learnable",
   "learnable",
   "learnable",
   "learnable"
  ],
  "thermometer": "distributive",
  "num_classes": 10,
  "tau": 3.3333333333333335,
  "batch_size": 100,
  "epochs": 30,
  "lr": 0.01,
  "lr_step": 14,
  "lr_gamma": 0.1,
  "seed": 20260811,
  "groupsum_group": 10,
  "note": "three steps. If this matches taper-2, depth past two steps buys nothing"
 },
 {
  "slug": "mnist_n6_z3_distributive_w1000x500x100",
  "label": "3x[1000, 500, 100]",
  "group": "taper-paper",
  "n": 6,
  "thermometer_bits": 3,
  "layers": [
   1000,
   500,
   100
  ],
  "mapping": [
   "learnable",
   "random",
   "learnable"
  ],
  "thermometer": "distributive",
  "num_classes": 10,
  "tau": 3.3333333333333335,
  "batch_size": 100,
  "epochs": 30,
  "lr": 0.01,
  "lr_step": 14,
  "lr_gamma": 0.1,
  "seed": 20260811,
  "groupsum_group": 10,
  "note": "taper on the paper's 2x[1000,500]; first two mappings match that baseline exactly"
 },
 {
  "slug": "mnist_n6_z3_distributive_w2000x300x200",
  "label": "3x[2000, 300, 200]",
  "group": "resolution",
  "n": 6,
  "thermometer_bits": 3,
  "layers": [
   2000,
   300,
   200
  ],
  "mapping": [
   "learnable",
   "learnable",
   "learnable"
  ],
  "thermometer": "distributive",
  "num_classes": 10,
  "tau": 3.3333333333333335,
  "batch_size": 100,
  "epochs": 30,
  "lr": 0.01,
  "lr_step": 14,
  "lr_gamma": 0.1,
  "seed": 20260811,
  "groupsum_group": 20,
  "note": "group 20 -- 21 score levels instead of 11"
 },
 {
  "slug": "mnist_n6_z3_distributive_w2000x300x500",
  "label": "3x[2000, 300, 500]",
  "group": "resolution",
  "n": 6,
  "thermometer_bits": 3,
  "layers": [
   2000,
   300,
   500
  ],
  "mapping": [
   "learnable",
   "learnable",
   "learnable"
  ],
  "thermometer": "distributive",
  "num_classes": 10,
  "tau": 3.3333333333333335,
  "batch_size": 100,
  "epochs": 30,
  "lr": 0.01,
  "lr_step": 14,
  "lr_gamma": 0.1,
  "seed": 20260811,
  "groupsum_group": 50,
  "note": "group 50 -- 51 score levels. If accuracy tracks resolution, the floor is the constraint"
 }
]

# The baselines these pair with, all in mnist_grid_kaggle.ipynb. Train those first or this
# notebook measures reduction against nothing.
BASELINES = {
    'mnist_n6_z3_distributive_w1000':      'group 100  -- pairs with 2x[1000, 100]',
    'mnist_n6_z3_distributive_w2000':      'group 200  -- pairs with the three 2000-node tapers',
    'mnist_n6_z3_distributive_w1000x500':  'group  50  -- pairs with 3x[1000, 500, 100]',
}

ONLY_SLUGS = None       # e.g. ['mnist_n6_z3_distributive_w2000x100'] to train one first
ONLY_N = None           # cap configs per session; None runs until the grid is done
PRECOMPUTE_LIMIT_GB = 6.0

print(f'{len(TRAINING_SET)} reduction configs')
for g in sorted({c['group'] for c in TRAINING_SET}):
    print(f"  {g:12s} {sum(1 for c in TRAINING_SET if c['group'] == g)}")
print()
print('GroupSum sees, per class:')
for c in TRAINING_SET:
    print(f"  {c['label']:28s} {c['groupsum_group']:>3d} bits -> "
          f"{c['groupsum_group'] + 1:>3d} score levels")


In [ ]:
# ---- load MNIST ----
# Matches third_party/DWN/examples/mnist.py: the canonical 60k/10k split, pixels in [0, 1].
# NO StandardScaler -- upstream uses transforms.ToTensor(), which is exactly /255. Our exporter
# never reads ck['scaler'], but the key is written below anyway so the checkpoint keeps the same
# shape as the JSC ones (mean 0, scale 255 reproduces this transform exactly).
import numpy as np, torch
from sklearn.datasets import fetch_openml

SEED = TRAINING_SET[0]['seed']
torch.manual_seed(SEED)
np.random.seed(SEED)

mnist = fetch_openml('mnist_784', version=1, as_frame=False)
X_all = mnist.data.astype(np.float32) / 255.0        # [0, 1], the ToTensor transform
y_all = mnist.target.astype(np.int64)
assert X_all.shape[1] == 784 and len(np.unique(y_all)) == 10

# The canonical split -- mnist_784 is ordered train-then-test, so this is the standard one every
# published MNIST number uses. Do not shuffle before slicing.
X_train, X_test = X_all[:60000], X_all[60000:]
y_train, y_test = y_all[:60000], y_all[60000:]

N_FEATURES = X_train.shape[1]
X_train_t = torch.from_numpy(X_train)
X_test_t = torch.from_numpy(X_test)
y_train_t = torch.from_numpy(y_train).long()
y_test_t = torch.from_numpy(y_test).long()
print('train', tuple(X_train_t.shape), ' test', tuple(X_test_t.shape))
print('pixel range', float(X_train_t.min()), '-', float(X_train_t.max()))
print(f'{(X_train_t == 0).float().mean():.1%} of training pixels are exactly zero -- so many '
      'quantile thresholds will coincide, and duplicate comparators cost nothing in hardware.')


In [ ]:
# ---- helpers ----
import time, os, gc
import torch_dwn as dwn
from torch import nn
from torch.nn.functional import cross_entropy

THERMOMETERS = {
    'plain': dwn.Thermometer,
    'linear': dwn.Thermometer,            # grid calls evenly-spaced 'linear'
    'gaussian': dwn.GaussianThermometer,
    'distributive': dwn.DistributiveThermometer,
}
WORK = '/kaggle/working'


def binarize_chunked(therm, x, chunk=10000):
    """Binarize -> flatten -> uint8 in slices, to bound peak memory (Phase 1 cell 6)."""
    out = torch.empty((x.size(0), x.size(1) * therm.num_bits), dtype=torch.uint8)
    for i in range(0, x.size(0), chunk):
        out[i:i + chunk] = therm.binarize(x[i:i + chunk]).flatten(start_dim=1).to(torch.uint8)
    return out


def make_group(kind, z):
    """Fit the thermometer and prepare batch access for every config sharing (kind, z)."""
    therm = THERMOMETERS[kind](z).fit(X_train_t)
    gb = (X_train_t.size(0) + X_test_t.size(0)) * N_FEATURES * z / 1e9
    thr_gpu = therm.thresholds.cuda()

    def on_the_fly(xf):
        # Same comparison binarization.py performs, done on the GPU so no huge CPU tensor
        # is ever materialized.
        return (xf.cuda().unsqueeze(-1) > thr_gpu).flatten(start_dim=1).float()

    if gb <= PRECOMPUTE_LIMIT_GB:
        xb_tr, xb_te = binarize_chunked(therm, X_train_t), binarize_chunked(therm, X_test_t)
        # The two paths MUST agree, or a large-z config would be trained on different bits
        # than a small-z one -- a difference that would look like a result.
        probe = on_the_fly(X_test_t[:256]).to(torch.uint8).cpu()
        assert torch.equal(probe, xb_te[:256]), 'on-the-fly binarization != precomputed'
        print(f'    precomputed {gb:.2f} GB uint8 (GPU path verified identical on 256 samples)')
        return therm, (lambda i: xb_tr[i].cuda().float()), (lambda i: xb_te[i].cuda().float()), \
            xb_tr.size(1), (xb_tr, xb_te)

    print(f'    on-the-fly binarization ({gb:.2f} GB would not fit)')
    return therm, (lambda i: on_the_fly(X_train_t[i])), (lambda i: on_the_fly(X_test_t[i])), \
        N_FEATURES * z, None


def train_one(cfg, get_train, get_test, in_bits):
    """Train one config. Returns (model, results dict)."""
    torch.manual_seed(cfg['seed'])
    layers, size = [], in_bits
    for i, w in enumerate(cfg['layers']):
        layers.append(dwn.LUTLayer(size, w, n=cfg['n'], mapping=cfg['mapping'][i]))
        size = w
    layers.append(dwn.GroupSum(k=cfg['num_classes'], tau=cfg['tau']))
    model = nn.Sequential(*layers).cuda()

    opt = torch.optim.Adam(model.parameters(), lr=cfg['lr'])
    sched = torch.optim.lr_scheduler.StepLR(opt, cfg['lr_step'], cfg['lr_gamma'])
    n_train, n_test = X_train_t.size(0), X_test_t.size(0)

    def evaluate(chunk=5000):
        model.eval()
        correct = 0
        with torch.no_grad():
            for i in range(0, n_test, chunk):
                sl = slice(i, i + chunk)
                correct += (model(get_test(sl)).argmax(1)
                            == y_test_t[sl].cuda()).sum().item()
        return correct / n_test

    best, history, losses = 0.0, [], []
    t0 = time.time()
    for ep in range(cfg['epochs']):
        model.train()
        perm = torch.randperm(n_train)
        run, nb = 0.0, 0
        for i in range(0, n_train, cfg['batch_size']):
            idx = perm[i:i + cfg['batch_size']]
            opt.zero_grad()
            loss = cross_entropy(model(get_train(idx)), y_train_t[idx].cuda())
            loss.backward()
            opt.step()
            run += loss.item(); nb += 1
        sched.step()
        losses.append(run / nb)
        acc = evaluate()
        history.append(acc); best = max(best, acc)
        if ep % 8 == 7 or ep == cfg['epochs'] - 1:
            print(f'      epoch {ep+1:3d}/{cfg["epochs"]}  loss {losses[-1]:.4f}  '
                  f'acc {acc:.4f}  [{time.time()-t0:.0f}s]')
    return model, {'final_acc': history[-1], 'best_acc': best,
                   'history': history, 'epoch_losses': losses,
                   'seconds': round(time.time() - t0, 1)}


def save_config(cfg, model, therm, res, get_test):
    """Checkpoint + 1000-sample test vectors, in the format exporter/extract.py reads."""
    slug = cfg['slug']
    ck_path = f'{WORK}/{slug}_checkpoint.pt'
    torch.save({
        'run_name': slug,
        # Exactly the keys the Phase 1 checkpoint carries -- extract.py reads config['n'],
        # ['layers'], ['thermometer_bits'], ['num_classes'] (docs/checkpoint-format.md).
        'config': {k: cfg[k] for k in (
            'thermometer', 'thermometer_bits', 'n', 'layers', 'mapping', 'num_classes',
            'tau', 'batch_size', 'epochs', 'lr', 'lr_step', 'lr_gamma', 'seed')},
        'pinned_commit': PINNED_COMMIT,
        'state_dict': model.state_dict(),
        'thermometer': {'kind': cfg['thermometer'], 'num_bits': cfg['thermometer_bits'],
                        'thresholds': therm.thresholds.cpu()},
        # x_scaled = (x_raw - 0) / 255, i.e. exactly what ToTensor does. Same shape as the
        # JSC checkpoints' StandardScaler entry so nothing downstream has to special-case it.
        'scaler': {'mean': torch.zeros(N_FEATURES),
                   'scale': torch.full((N_FEATURES,), 255.0)},
        'classes': [str(i) for i in range(10)],
        'feature_names': [f'pixel{i}' for i in range(N_FEATURES)],
        'results': res,
        'grid_label': cfg['label'],
        'torch_version': torch.__version__,
    }, ck_path)

    # Gate 1 needs these: gen_vectors.py reads <run>_testvectors.npz beside the checkpoint.
    # x_binarized drives the core testbench, x_raw the encoder+core one -- both, so a Gate 1
    # failure localizes to one or the other.
    model.eval()
    with torch.no_grad():
        xb = get_test(slice(0, 1000))
        pred = model(xb).argmax(1).cpu().numpy()
    np.savez_compressed(
        f'{WORK}/{slug}_testvectors.npz',
        x_binarized=xb.to(torch.uint8).cpu().numpy(),
        x_raw=X_test[:1000], y=y_test[:1000], pred=pred)
    return ck_path

In [ ]:
# ---- carry forward anything already trained in an EARLIER SESSION ----
# /kaggle/working is fresh on every version -- it only persists while one session is alive.
# So the resume check below would restart from zero on a new session, which is precisely the
# case a 32-config run needs. Add the previous run's OUTPUT as an input dataset and this copies
# it forward, so the final Output panel holds the complete set rather than one session's slice.
import glob, shutil

carried = 0
for src in glob.glob('/kaggle/input/**/*_checkpoint.pt', recursive=True) + \
           glob.glob('/kaggle/input/**/*_testvectors.npz', recursive=True):
    dst = os.path.join(WORK, os.path.basename(src))
    if not os.path.exists(dst):
        shutil.copy(src, dst)
        carried += 1
print(f'carried forward {carried} file(s) from /kaggle/input')
if not carried:
    print('(none -- first session, or no previous output added as an input dataset)')

In [ ]:
# ---- train the grid ----
# Ordered by (encoding, z) so each binarization is built once and reused by every config that
# shares it. Within a group, configs run cheapest-first so a session that dies late still
# banks the most models.
import collections

# Filter FIRST. The training loop below iterates by_group, not TRAINING_SET, so restricting
# TRAINING_SET after by_group is built changes only the printed counts and trains the whole grid
# anyway -- lowest z first, which is not the config anyone asked for.
if ONLY_SLUGS:
    TRAINING_SET = [c for c in TRAINING_SET if c['slug'] in ONLY_SLUGS]
    assert TRAINING_SET, 'ONLY_SLUGS matched nothing -- check the slug spelling'
    print(f'ONLY_SLUGS: restricted to {len(TRAINING_SET)} config(s): '
          + ', '.join(c['slug'] for c in TRAINING_SET))

by_group = collections.defaultdict(list)
for c in TRAINING_SET:
    by_group[(c['thermometer'], c['thermometer_bits'])].append(c)
for v in by_group.values():
    v.sort(key=lambda c: sum(c['layers']))

done = [c for c in TRAINING_SET if os.path.exists(f"{WORK}/{c['slug']}_checkpoint.pt")]
todo = [c for c in TRAINING_SET if c not in done]
print(f'{len(done)} already trained, {len(todo)} to go')
if ONLY_N:
    print(f'ONLY_N={ONLY_N}: stopping after {ONLY_N} this session')

summary, trained = [], 0
for (kind, z), configs in sorted(by_group.items(), key=lambda kv: kv[0][1]):
    pending = [c for c in configs
               if not os.path.exists(f"{WORK}/{c['slug']}_checkpoint.pt")]
    if not pending or (ONLY_N and trained >= ONLY_N):
        continue
    print(f'\n=== {kind}, z={z} -- {len(pending)} config(s) ===')
    therm, get_train, get_test, in_bits, held = make_group(kind, z)

    for cfg in pending:
        if ONLY_N and trained >= ONLY_N:
            break
        print(f'  -- {cfg["slug"]}  ({cfg["label"]}, {sum(cfg["layers"])} nodes)')
        model, res = train_one(cfg, get_train, get_test, in_bits)
        path = save_config(cfg, model, therm, res, get_test)
        print(f'     final {res["final_acc"]:.4f}  best {res["best_acc"]:.4f}  '
              f'{res["seconds"]:.0f}s  -> {os.path.basename(path)}')
        summary.append((cfg['label'], cfg['slug'], res['final_acc'], res['seconds']))
        trained += 1
        del model; gc.collect(); torch.cuda.empty_cache()

    del therm, get_train, get_test, held
    gc.collect(); torch.cuda.empty_cache()

print(f'\ntrained {trained} config(s) this session')

In [ ]:
# ---- summary ----
import glob
cks = sorted(glob.glob(f'{WORK}/*_checkpoint.pt'))
print(f'{len(cks)} checkpoints in {WORK} ({len(TRAINING_SET)} configs in the grid)')
print()
if summary:
    print(f'{"config":24s} {"final acc":>10} {"seconds":>9}')
    print('-' * 46)
    for label, slug, acc, secs in summary:
        print(f'{label:24s} {100*acc:>9.2f}% {secs:>9.0f}')
    print('-' * 46)

missing = [c['slug'] for c in TRAINING_SET
           if not os.path.exists(f"{WORK}/{c['slug']}_checkpoint.pt")]
if missing:
    print(f'\nSTILL MISSING ({len(missing)}): re-run this notebook to continue.')
    for s in missing[:10]:
        print('  ', s)
    if len(missing) > 10:
        print(f'   ... and {len(missing)-10} more')
else:
    print('\nAll configs trained. Download every *_checkpoint.pt AND *_testvectors.npz')
    print('into training/artifacts/, then run:  python dse/run.py --all --impl')

## After this runs

Download **both** files per config into `training/artifacts/mnist/`, then put them through the
normal flow — no new RTL is needed, because `rtlgen/emit_core.py` already loops over layers and
multi-layer emission is Gate 1 verified.

```
python scripts/run_gate1.py --checkpoint training/artifacts/mnist/<slug>_checkpoint.pt
python scripts/run_synth.py --rtl-dir build/rtl
```

**Gate 1 first, always.** A taper changes the shape of the core, and area for unverified RTL
describes nothing.

## Reading the result

Pair each config against its baseline and compare three things:

| | what it tells you |
|---|---|
| **accuracy vs the baseline** | whether a learned taper preserves what counting votes achieves |
| **`dwn_core` LUTs** | the taper's real cost: extra nodes against a much smaller adder tree |
| **Fmax** | the popcount was the critical path on JSC; a shorter tree should move it |

A taper that holds accuracy and shrinks the core is the result this notebook exists to find. A
taper that loses accuracy is equally publishable — it would mean the fixed vote-count is a
stronger inductive bias than a learned combination, which nobody has measured either.

⚠️ **Accuracy differences below the noise floor are not differences.** The JSC study measured
run-to-run spread at 0.15 pp by training identical configurations twice. No equivalent figure has
been measured for MNIST yet; until one is, treat small gaps with suspicion rather than confidence.
